In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

## Load and Clean Corporate Bond Data

In [32]:
df = pd.read_excel("./data/cleaneddata.xlsx")
df.columns = df.columns.str.strip()

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values(["cusip","date"])

for col in ["spread","price","sduration","coupon","ytm"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["date","spread","price","sduration"])
df["spread"] = df["spread"] / 10000.0

df = df[df["date"].between("2017-01-01","2024-12-06")]
df["ym"] = df["date"].dt.to_period("M")

## Build Monthly Panel and Compute Returns and DTS

In [36]:
price_col = "price"

first_day = (
    df.sort_values("date")
      .groupby(["cusip","ym"])
      .first()
      .reset_index()
)

# next month price + next month label
first_day["next_price"] = first_day.groupby("cusip")[price_col].shift(-1)
first_day["next_ym"]    = first_day.groupby("cusip")["ym"].shift(-1)

first_day = first_day.dropna(subset=["next_price","next_ym"])

# enforce valid month pairs
first_day = first_day[
    first_day["ym"].between("2017-01","2024-12") &
    first_day["next_ym"].between("2017-01","2024-12")
]

first_day["price_ret"] = (
    (first_day["next_price"] - first_day[price_col]) /
     first_day[price_col]
)

# carry = coupon%/12 / price
first_day["carry"] = ((first_day["coupon"] / 100) / 12) / first_day[price_col]

# total return
first_day["ret"] = first_day["price_ret"] + first_day["carry"]

# DTS (for sorting into buckets; scaling doesn't affect order)
first_day["dts"] = first_day["spread"] * first_day["sduration"]

# wide monthly return matrix
ret_pivot = (
    first_day.pivot(index="ym", columns="cusip", values="ret")
    .sort_index()
)

## Portfolio Optimizer (Long-Only Maximum Sharpe Ratio)

In [21]:
def optimize_weights(mu, cov, w_max=0.05):
    """Maximize Sharpe ratio with weight caps (long-only)."""
    n = len(mu)
    w0 = np.ones(n) / n

    def neg_sharpe(w):
        ret = np.dot(w, mu)
        vol = np.sqrt(np.dot(w.T, np.dot(cov, w)))
        if vol <= 0:
            return 1e9
        return -(ret / vol)

    bounds = [(0, w_max)] * n
    cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]

    res = minimize(
        neg_sharpe, w0,
        method="SLSQP",
        bounds=bounds,
        constraints=cons,
        options={"maxiter": 500, "ftol": 1e-9}
    )

    return res.x

## Build Unhedged Portfolios

In [38]:
port_low  = []
port_mid  = []
port_high = []

months = sorted(first_day["ym"].unique())

for idx in range(3, len(months)):
    ym = months[idx]

    # trailing 3 months for mean/cov estimation
    window = months[idx-3:idx]
    hist = ret_pivot.loc[window]

    # current month bonds
    month_df = first_day[first_day["ym"] == ym].copy()
    month_df = month_df.sort_values("dts")
    n = len(month_df)
    if n < 6:
        continue

    # split into 3 equal DTS buckets
    k = n // 3
    low  = month_df.iloc[:k].copy()
    mid  = month_df.iloc[k:2*k].copy()
    high = month_df.iloc[2*k:3*k].copy()

    # helper
    def prepare_bucket(bucket):
        cus = list(bucket["cusip"])
        sub = hist[cus].dropna(axis=1, how="any")

        if sub.shape[1] < 2:
            return None, None, None

        cusips = list(sub.columns)
        bucket_ordered = bucket.set_index("cusip").loc[cusips].reset_index()

        mu  = sub.mean().values
        cov = np.cov(sub.T)

        return bucket_ordered, mu, cov

    # prepare each
    low_b,  mu_low,  cov_low  = prepare_bucket(low)
    mid_b,  mu_mid,  cov_mid  = prepare_bucket(mid)
    high_b, mu_high, cov_high = prepare_bucket(high)

    if low_b is None or mid_b is None or high_b is None:
        continue

    # optimize
    w_low  = optimize_weights(mu_low,  cov_low)
    w_mid  = optimize_weights(mu_mid,  cov_mid)
    w_high = optimize_weights(mu_high, cov_high)

    # realized return
    ret_low  = np.dot(w_low,  low_b["ret"].values)
    ret_mid  = np.dot(w_mid,  mid_b["ret"].values)
    ret_high = np.dot(w_high, high_b["ret"].values)

    # portfolio sduration (for Treasury hedge)
    dur_low  = np.dot(w_low,  low_b["sduration"].values)
    dur_mid  = np.dot(w_mid,  mid_b["sduration"].values)
    dur_high = np.dot(w_high, high_b["sduration"].values)

    # append
    port_low.append({
        "month": ym,
        "ret": ret_low,
        "dur_p": dur_low,
        "cusips": list(low_b["cusip"]),
    })

    port_mid.append({
        "month": ym,
        "ret": ret_mid,
        "dur_p": dur_mid,
        "cusips": list(mid_b["cusip"]),
    })

    port_high.append({
        "month": ym,
        "ret": ret_high,
        "dur_p": dur_high,
        "cusips": list(high_b["cusip"]),
    })

low_opt  = pd.DataFrame(port_low)
mid_opt  = pd.DataFrame(port_mid)
high_opt = pd.DataFrame(port_high)

# cumulative returns (unhedged)
low_opt["cum_ret"]  = (1 + low_opt["ret"]).cumprod() - 1
mid_opt["cum_ret"]  = (1 + mid_opt["ret"]).cumprod() - 1
high_opt["cum_ret"] = (1 + high_opt["ret"]).cumprod() - 1

print("LOW DTS (unhedged):")
print(low_opt.tail(), "\n")

print("MID DTS (unhedged):")
print(mid_opt.tail(), "\n")

print("HIGH DTS (unhedged):")
print(high_opt.tail(), "\n")

LOW DTS (unhedged):
      month       ret   dur_p  \
87  2024-07  0.011720  1.8950   
88  2024-08  0.004532  1.9385   
89  2024-09 -0.002873  1.8660   
90  2024-10 -0.003898  1.7265   
91  2024-11  0.003039  1.3735   

                                               cusips   cum_ret  
87  [459200JG7, 25468PDK9, 191216CE8, 822582BX9, 4... -0.043230  
88  [25468PDK9, 459200JG7, 46132FAD2, 191216CE8, 2... -0.038893  
89  [25468PDK9, 459200JG7, 46132FAD2, 822582BX9, 1... -0.041654  
90  [25468PDK9, 191216CE8, 459200JG7, 64110LAN6, 0... -0.045390  
91  [191216CE8, 459200JG7, 25468PDK9, 64110LAN6, 0... -0.042489   

MID DTS (unhedged):
      month       ret   dur_p  \
87  2024-07  0.025132  9.0470   
88  2024-08  0.016778  9.5395   
89  2024-09 -0.008422  9.5355   
90  2024-10 -0.033096  9.5885   
91  2024-11  0.026427  8.7215   

                                               cusips   cum_ret  
87  [219350AX3, 126408GS6, 45687AAG7, 428236BR3, 0... -0.078886  
88  [12527GAF0, 126408GS6, 45687

## Load Treasury Returns

In [40]:
treasury = pd.read_csv("./data/treasury_data.csv")
treasury["ym"] = treasury["ym"].astype("period[M]")
treasury = treasury.set_index("ym").sort_index()

# Assume a constant Treasury duration (e.g. IEF ~ 7.5 years)
duration_T = 7.5

## Merge Portfolios with Treasury and Compute Hedge

In [42]:
for df_opt in [low_opt, mid_opt, high_opt]:
    df_opt["ym"] = df_opt["month"]
    df_opt.set_index("ym", inplace=True)

# Join Treasury returns
low_opt  = low_opt.join(treasury, how="inner")
mid_opt  = mid_opt.join(treasury, how="inner")
high_opt = high_opt.join(treasury, how="inner")

# Hedge ratios: theta_T = -Dur_portfolio / Dur_Treasury
low_opt["theta_T"]  = -low_opt["dur_p"]  / duration_T
mid_opt["theta_T"]  = -mid_opt["dur_p"]  / duration_T
high_opt["theta_T"] = -high_opt["dur_p"] / duration_T

# Duration-hedged returns
low_opt["ret_hedged"]  = low_opt["ret"]  + low_opt["theta_T"] * low_opt["ret_T"]
mid_opt["ret_hedged"]  = mid_opt["ret"]  + mid_opt["theta_T"] * mid_opt["ret_T"]
high_opt["ret_hedged"] = high_opt["ret"] + high_opt["theta_T"] * high_opt["ret_T"]

# Cumulative duration-hedged returns
low_opt["cum_hedged"]  = (1 + low_opt["ret_hedged"]).cumprod() - 1
mid_opt["cum_hedged"]  = (1 + mid_opt["ret_hedged"]).cumprod() - 1
high_opt["cum_hedged"] = (1 + high_opt["ret_hedged"]).cumprod() - 1

print("LOW DTS – Duration-Hedged with Treasuries:")
print(low_opt[["month","ret","dur_p","ret_T","theta_T","ret_hedged","cum_hedged"]].tail(), "\n")

print("MID DTS – Duration-Hedged with Treasuries:")
print(mid_opt[["month","ret","dur_p","ret_T","theta_T","ret_hedged","cum_hedged"]].tail(), "\n")

print("HIGH DTS – Duration-Hedged with Treasuries:")
print(high_opt[["month","ret","dur_p","ret_T","theta_T","ret_hedged","cum_hedged"]].tail())

LOW DTS – Duration-Hedged with Treasuries:
           month       ret   dur_p     ret_T   theta_T  ret_hedged  cum_hedged
ym                                                                            
2024-07  2024-07  0.011720  1.8950  0.028901 -0.252667    0.004418   -0.086029
2024-08  2024-08  0.004532  1.9385  0.013493 -0.258467    0.001045   -0.085075
2024-09  2024-09 -0.002873  1.8660  0.013867 -0.248800   -0.006323   -0.090859
2024-10  2024-10 -0.003898  1.7265 -0.033823 -0.230200    0.003888   -0.087325
2024-11  2024-11  0.003039  1.3735  0.010033 -0.183133    0.001202   -0.086228 

MID DTS – Duration-Hedged with Treasuries:
           month       ret   dur_p     ret_T   theta_T  ret_hedged  cum_hedged
ym                                                                            
2024-07  2024-07  0.025132  9.0470  0.028901 -1.206267   -0.009730   -0.149721
2024-08  2024-08  0.016778  9.5395  0.013493 -1.271933   -0.000384   -0.150048
2024-09  2024-09 -0.008422  9.5355  0.01386

## Compute Empirical VaR and ES

In [65]:
low_monthly = np.array(low_opt[['ret_hedged']])
mid_monthly = np.array(mid_opt[['ret_hedged']])
high_monthly = np.array(high_opt[['ret_hedged']])

low_emp_var = np.percentile(low_monthly,0.05)
mid_emp_var = np.percentile(mid_monthly,0.05)
high_emp_var = np.percentile(high_monthly,0.05)

low_exced = low_monthly[low_monthly < low_emp_var]
low_emp_es = low_exced.mean()
mid_exced = mid_monthly[mid_monthly < mid_emp_var]
mid_emp_es = mid_exced.mean()
high_exced = high_monthly[high_monthly < high_emp_var]
high_emp_es = high_exced.mean()

print('1-Month Empirical VaR and ES:')
print(f'Low DTS: Var = {low_emp_var:0.4f}; ES = {low_emp_es:0.4f}')
print(f'Mid DTS: Var = {mid_emp_var:0.4f}; ES = {mid_emp_es:0.4f}')
print(f'High DTS: Var = {high_emp_var:0.4f}; ES = {high_emp_es:0.4f}')


Empirical VaR and ES:
Low DTS: Var = -0.0660; ES = -0.0684
Mid DTS: Var = -0.1283; ES = -0.1328
High DTS: Var = -0.2140; ES = -0.2217


## Empirical Spectral Risk Measure (Exponentially Weighted Losses)

In [78]:
low_losses = low_monthly * -1
mid_losses = mid_monthly * -1
high_losses = high_monthly * -1

def spectral_risk_measure(losses, phi):
    losses = np.sort(losses)
    n = len(losses)
    weights = np.zeros(n)
    
    for k in range(1, n+1):
        a = (k-1)/n
        b = k/n
        # numeric integration of phi(u) on [a,b]
        u = np.linspace(a, b, 50)
        weights[k-1] = np.trapz(phi(u), u)

    return np.sum(losses * weights)

gamma = 20
phi = lambda u: gamma * np.exp(gamma*u) / (np.exp(gamma)-1)

low_spectral = spectral_risk_measure(low_losses, phi)
mid_spectral = spectral_risk_measure(mid_losses, phi)
high_spectral = spectral_risk_measure(high_losses, phi)

print(low_spectral)
print(mid_spectral)
print(high_spectral)

0.08552830991955289
0.14053305627170767
0.05167468629279601


In [100]:
qs = np.linspace(0.95, 0.999, 20)   # focus on the tail
VaR = np.quantile(high_losses, qs)

tail_df = pd.DataFrame({"q": qs, "VaR": VaR})

High q= 0.1 VaR(losses)≈ -0.018857108218759393
High q= 0.3 VaR(losses)≈ -0.008403055465482373
High q= 0.5 VaR(losses)≈ 9.407345972034173e-05
High q= 0.7 VaR(losses)≈ 0.009314531069192505
High q= 0.9 VaR(losses)≈ 0.023130791616580892
